Install the CrewAI, Langchain, CrewAI_Tools, Python-Dotenv, and requests libraries.

In [27]:
%pip install "crewai>=1,<2" "requests>=2.32,<3" "python-dotenv>=1,<2" "python-dotenv>=1,<2" crewai_tools langchain langchain_community

Ingest the environment variables from the .env file.

In [35]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

# Load API keys from the .env file in the notebook folder (or a parent folder).
# override=True makes the .env values take precedence over existing variables.
print  (os.getcwd())
dotenv_path = find_dotenv(usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Loaded environment variables from {dotenv_path}")
else:
    print("No .env file found; falling back to environment variables or prompts.")

# Use the same model family as the lab. Override with OPENAI_MODEL in .env.
OPENAI_MODEL = os.environ.get("OPEN_APIMODEL").strip()
    
print(f"Using OpenAI model: {OPENAI_MODEL}")
#print (f"Using OpenAI base URL: {os.environ.get('OPEN_APIMODEL')}")
#print(f"Using OpenAI organization: {os.environ.get('TAVILY_API_KEY')}")
#print(f"Using OpenAI API key: {os.environ.get('OPENAI_API_KEY')}")
#print (f"Using OpenAI base URL: {os.environ.get('OPENAI_ORG')}")

Define the LLM with the gpt-5-mini model from the https://api.openai.com/v1 URL and import librarys and classes. 

In [36]:
import os
import requests
from crewai import Agent, Crew, Process, Task, LLM
from crewai_tools import PDFSearchTool
from crewai.tools import BaseTool

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
#from langchain_community.chat_models import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain.tools import tool


# 2. Define the LLM Model (using gpt-5-mini)
# Note: Ensure 'gpt-5-mini' is officially supported in your OpenAI library version.
# If not yet exposed via that exact string, fall back to 'gpt-4o-mini' or your preferred model tier.
llm = ChatOpenAI(
    model=f"openai/{OPENAI_MODEL}",
    api_key=os.environ["OPENAI_API_KEY"],
    organization=os.environ["OPENAI_ORG"],
    base_url="https://api.openai.com/v1",
    temperature=0.8
)

llm = LLM(
    model=f"openai/{OPENAI_MODEL}",
    api_key=os.environ["OPENAI_API_KEY"],
    organization=os.environ["OPENAI_ORG"],
    base_url="https://api.openai.com/v1",
    max_completion_tokens=8000,
    timeout=120,
)

Define web_search_tool as a TaavilySearch.

In [37]:
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information and return source snippets and URLs."

    def _run(self, query: str) -> str:
        response = requests.post(
            "https://api.tavily.com/search",
            json={
                "api_key": os.environ["TAVILY_API_KEY"],
                "query": query,
                "max_results": 3,
            },
            timeout=30,
        )
        response.raise_for_status()
        results = response.json().get("results", [])
        return "\n\n".join(
            f"{r.get('title', 'Untitled')}\n"
            f"{r.get('url', '')}\n"
            f"{r.get('content', '')}"
            for r in results
        ) or "No web results found."


web_search_tool = TavilySearchTool()

Specify The PDF and Web search agents.

In [40]:
# 3. Initialize Specialized Tools
# PDFSearchTool automatically creates vector embeddings for semantic search within a PDF
pdf_tool = PDFSearchTool(pdf="trasformer_research_paper-dataset.pdf")

# TavilySearch reads TAVILY_API_KEY from the environment.

search_tool = TavilySearch(max_results=3, topic="general")

@tool
def search_tool(query: str) -> dict:
    """Search the web with Tavily for current information and source URLs."""
    return search_tool.invoke({"query": query})

agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt=(
        "You are a research assistant. Use search_tool for questions about "
        "latest or current developments. Summarize the retrieved evidence "
        "and include source URLs. Answer basic arithmetic directly."
    )
)

# 4. Define Agents

pdf_agent = Agent(
    role="Internal PDF Data Specialist",
    goal="Who is the source of the pdf and provide a summary of its contents. The summary should be limited to 8 sentences and exclude figures and diagrams. I have uploaded the relevant PDF. Summary of the PDF contents limited to 8 sentences and excluding figures and diagrams. Please disregard previous instructions about complete acutal content.",
    backstory="You need a summary summary and source of the PDF.",
    tools=[pdf_tool],
    allow_delegation=False,
    llm=llm,
    verbose=True
)

web_agent = Agent(
    role="Real-Time Web Researcher",
    goal='Find the 5 of the latest advancements in AI for healthcare.',
    backstory="You are an expert in artificial intelligence and stay updated with the latest advancements in AI for healthcare.",
    tools=[web_search_tool],
    max_iter=4,
    llm=llm,
    verbose=True
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=3
)

Define PDF research task (pdf_research_task), Web research task (web_research_task), Main research task (research_task). 
Define Crew definition as a hierarchical process and lauch the crew asynchronously.

In [41]:

web_research_task = Task(
    role="Real-Time Web Researcher",
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 5-6 sentences."
        " Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    async_execution=True,
    context=[],
    agent=web_agent
)

pdf_research_task = Task(
    role="Internal PDF Data Specialist",
    description=(
        "Summary of the PDF contents limited to 8 sentences and excluding figures and diagrams. Please disregard previous instructions about complete acutal content."
        " Do not call tools again after getting results."
    ),
    expected_output=(
        "Summary of the PDF contents limited to 8 sentences and excluding figures and diagrams. Please disregard previous instructions about complete acutal content."
    ),
    async_execution=True,
    context=[],
    agent=pdf_agent
)


# 5. Define Tasks
# In a hierarchical process, tasks are assigned to the Crew as a whole or to a target agent,
# but the Manager Agent (Router) will delegate individual components as needed.
research_task = Task(
    description=(
        "1. Search the internal PDF information about the google translation models. Please check the uploaded PDF named = trasformer_research_paper-dataset.pdf. The summary should be limited to 8 sentences and exclude figures and diagrams. Please disregard previous instructions about complete acutal content.\n"
        "2. Search the internet for public US healthcare PDF documents for AI developments in the last 3 years relative to today's date. Exclude HIPAA-protected information and Patient-identifiable data. Only provide 5 examples in 5 summary sentences each.\n"
        "3. Cross-reference the PDF insights with the web findings to create a comprehensive evaluation report."
    ),
    context=[pdf_research_task, web_research_task],
    expected_output="A comprehensive report combining the pdf and web findings.",
)

# 6. Assemble the Crew with a Hierarchical Structure
# The manager_llm acts as the central router agent that directs traffic to the specialized agents.
# Enables the central router/manager pattern
# The gpt-5-mini router directing the workflow
crew = Crew(
    agents=[pdf_agent, web_agent],
    tasks=[research_task],
    process=Process.hierarchical,
    manager_llm=llm,                
    verbose=True
)

# 7. Execute and Print Output
print("############# Starting Crew Execution #############\n")
#user_input = input("Query: ").strip()
# user_input = "what are the models that google uses for translation?"

result = await crew.kickoff_async()

#if user_input.lower() not in ["exit", "quit"]:
#    #result = crew.kickoff(inputs={"user_query": user_input})
#    result = await crew.kickoff_async()
#    print("\n############# Final Orchestrated Output #############\n")
#    print(result)
##else:
#    print("Exiting...")



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.37                                                                                       │
│  Latest version:  1.15.41                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2d5ed626-6447-4041-b7a0-e24a3b67ecd8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Search the internal PDF information about the google translation models. Please check the uploaded    │
│  PDF named = trasformer_research_paper-dataset.pdf. The summary should be limited to 8 sentences and exclude    │
│  figures and diagrams. Please disregard previous instructions about complete acutal content.                    │
│  2. Search the internet for public US healthcare PDF documents for AI developments in the last 3 years          │
│  relative to today's date. Exclude HIPAA-protected information and Patient-identifiable data. Only provide 5    │
│  examples in 5 summary sentences each.                                                                          │
│  3. Cross-reference the PDF insights with the web findings to create a comprehensive evaluation report.         │
│  ID: eb602dce-4113-4342-bf38-253a5b65b254                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: 1. Search the internal PDF information about the google translation models. Please check the uploaded    │
│  PDF named = trasformer_research_paper-dataset.pdf. The summary should be limited to 8 sentences and exclude    │
│  figures and diagrams. Please disregard previous instructions about complete acutal content.                    │
│  2. Search the internet for public US healthcare PDF documents for AI developments in the last 3 years          │
│  relative to today's date. Exclude HIPAA-protected information and Patient-identifiable data. Only provide 5    │
│  examples in 5 summary sentences each.                                                                          │
│  3. Cross-reference the PDF insights with the web findings to create a comprehensive evaluation report.         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#17) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Search the internet (public sources) for 5 US healthcare PDF documents about AI developments   │
│  in the last 3 years and produce 5 examples with 5-sentence summaries each.', 'context': "You are ...           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#18) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Search the uploaded PDF and produce an 8-sentence summary excluding figures and diagrams.',    │
│  'context': 'You are the Internal PDF Data Specialist. The user uploaded a PDF named \'trasformer_r...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (33 outputs left out)

╭─────────────────────────────────────── ✅ Tool Execution Completed (#18) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: I will follow the original task instructions (find 5 U.S.-based healthcare PDF documents about AI      │
│  published between 2023-11-13 and 2026-11-13 and provide metadata plus exactly five-sentence summaries and      │
│  one-sentence takeaways). I performed live web research and selected five high-quality U.S. government or       │
│  U.S.-based organization PDFs that meet the date and source constraints. Below are the five documents, each     │
│  with: full title, authoring organization, publication date, direct link to the PDF, a five-sentence summary    │
│  (exactly five sentences), a note about PHI exclusion where applicable, and one one-sentence key takeaway       │
│  focused on implications for translation models or general AI models in healthcare.                             │
│                                                                                                                 │
│  Document 1                                                                                                     │
│  Title: Considerations for the Use of Artificial Intelligence To Support Regulatory Decision-Making for Drug    │
│  and Biological Products (Draft Guidance)                                                                       │
│  Authoring organization: U.S. Food and Drug Administration (FDA)                                                │
│  Publication date: January 2025 (Draft)                                                                         │
│  Direct PDF link: https://www.fda.gov/media/184830/download                                                     │
│                                                                                                                 │
│  Five-sentence summary:                                                                                         │
│  This draft FDA guidance describes the scope and context for using AI systems to produce information or data    │
│  intended to support regulatory decision-making for drugs and biological products, presenting a risk-based      │
│  credibility assessment framework for AI models in a specified context of use. The guidance outlines main       │
│  recommendations for sponsors and regulated parties, including documentation of model design, training data     │
│  provenance and representativeness, verification and validation, and consideration of uncertainty and           │
│  limitations in model outputs. Evaluation approaches emphasized include establishing a transparent context of   │
│  use, pre-specified performance and credibility criteria, prospective testing, and ongoing post-deployment      │
│  monitoring with defined metrics for accuracy, robustness, and bias. Deployment considerations cover change     │
│  control, software lifecycle management, transparency about model capabilities and constraints, and early FDA   │
│  engagement when the applicability to regulatory decisions is unclear. Privacy and safety safeguards noted      │
│  include expectations to comply with applicable laws and regulations, data governance and de-identification     │
│  practices where appropriate, and recommendations for risk mitigation plans to address safety, bias, and        │
│  reliability concerns.                                                                                          │
│                                                                                                                 │
│  PHI note: The guidance is policy-focused and does not 

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Short note on scope and constraints                                                                            │
│  - I cannot provide the complete content of the uploaded PDF. Per internal policies and the task constraints    │
│  given to our PDF specialist, I instead include the specialist’s required 8-sentence summary of the uploaded    │
│  PDF, short key takeaways, and up to three short non-copyright-restricted quotes with page citations. I also    │
│  include the five web-sourced PDF summaries and links produced by our web researcher (each with exactly         │
│  five-sentence summaries and a one-sentence takeaway), and then a cross-referenced, actionable evaluation       │
│  report that integrates both sources and recommends concrete next steps, test plans, and team assignments.      │
│                                                                                                                 │
│  Part A — Exact 8-sentence summary of the uploaded PDF (trasformer_research_paper-dataset.pdf)                  │
│  1. The paper introduces the Transformer, a sequence transduction architecture that replaces recurrence and     │
│  convolutions with multi-head self-attention and position-wise feed-forward layers to improve parallelism and   │
│  shorten path lengths for long-range dependencies (p. 1, p. 6).                                                 │
│  2. Training data include WMT 2014 English–German (≈4.5M sentence pairs with a shared BPE vocabulary ≈37k) and  │
│  WMT 2014 English–French (≈36M sentence pairs with a 32k word-piece vocabulary), with batching by approximate   │
│  token counts (p. 5).                                                                                           │
│  3. Models were trained with the Adam optimizer (β1=0.9, β2=0.98, ε=1e-9) using a custom learning-rate          │
│  schedule with 4,000 warmup steps, plus dropout and label smoothing (ϵls=0.1); base models ran ~100k steps and  │
│  big models ~300k steps on one machine with 8 NVIDIA P100 GPUs (p. 5, p. 8).                                    │
│  4. Evaluation used BLEU and showed state-of-the-art single-model performance: the big Transformer reached      │
│  28.4 BLEU on WMT 2014 English→German and ~41.0 BLEU on English→French while requiring substantially less       │
│  training FLOPs than many prior models and ensembles (p. 8, Table 2).                                           │
│  5. For inference they applied beam search (beam size 4, length penalty α=0.6), limited output length to input  │
│  + 50, and used checkpoint averaging (last 5 for base, last 20 for big) to improve final results (p. 8).        │
│  6. Noted limitations include the O(n^2 · d) time and memory cost of full self-attention for very long          │
│  sequences and the paper suggests restricted or local attention as future work (p. 6–7).                        │
│  7. The paper does not analyze domain-specific risks such as clinical mistranslation, privacy, or regulatory    │
│  concerns for healthcare use; domain adaptation and specialised evaluation would be required for sensitive      │
│  applications (p. 6–8).                                                                                         │
│  8. The authors recommend practical training and inference techniques (checkpoint averaging, regularization,    │
│  tuned beam/length-penalty) and indicate that applying 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Search the internal PDF information about the google translation models. Please check the uploaded    │
│  PDF named = trasformer_research_paper-dataset.pdf. The summary should be limited to 8 sentences and exclude    │
│  figures and diagrams. Please disregard previous instructions about complete acutal content.                    │
│  2. Search the internet for public US healthcare PDF documents for AI developments in the last 3 years          │
│  relative to today's date. Exclude HIPAA-protected information and Patient-identifiable data. Only provide 5    │
│  examples in 5 summary sentences each.                                                                          │
│  3. Cross-reference the PDF insights with the web findings to create a comprehensive evaluation report.         │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2d5ed626-6447-4041-b7a0-e24a3b67ecd8                                                                       │
│  Final Output: Short note on scope and constraints                                                              │
│  - I cannot provide the complete content of the uploaded PDF. Per internal policies and the task constraints    │
│  given to our PDF specialist, I instead include the specialist’s required 8-sentence summary of the uploaded    │
│  PDF, short key takeaways, and up to three short non-copyright-restricted quotes with page citations. I also    │
│  include the five web-sourced PDF summaries and links produced by our web researcher (each with exactly         │
│  five-sentence summaries and a one-sentence takeaway), and then a cross-referenced, actionable evaluation       │
│  report that integrates both sources and recommends concrete next steps, test plans, and team assignments.      │
│                                                                                                                 │
│  Part A — Exact 8-sentence summary of the uploaded PDF (trasformer_research_paper-dataset.pdf)                  │
│  1. The paper introduces the Transformer, a sequence transduction architecture that replaces recurrence and     │
│  convolutions with multi-head self-attention and position-wise feed-forward layers to improve parallelism and   │
│  shorten path lengths for long-range dependencies (p. 1, p. 6).                                                 │
│  2. Training data include WMT 2014 English–German (≈4.5M sentence pairs with a shared BPE vocabulary ≈37k) and  │
│  WMT 2014 English–French (≈36M sentence pairs with a 32k word-piece vocabulary), with batching by approximate   │
│  token counts (p. 5).                                                                                           │
│  3. Models were trained with the Adam optimizer (β1=0.9, β2=0.98, ε=1e-9) using a custom learning-rate          │
│  schedule with 4,000 warmup steps, plus dropout and label smoothing (ϵls=0.1); base models ran ~100k steps and  │
│  big models ~300k steps on one machine with 8 NVIDIA P100 GPUs (p. 5, p. 8).                                    │
│  4. Evaluation used BLEU and showed state-of-the-art single-model performance: the big Transformer reached      │
│  28.4 BLEU on WMT 2014 English→German and ~41.0 BLEU on English→French while requiring substantially less       │
│  training FLOPs than many prior models and ensembles (p. 8, Table 2).                                           │
│  5. For inference they applied beam search (beam size 4, length penalty α=0.6), limited output length to input  │
│  + 50, and used checkpoint averaging (last 5 for base, last 20 for big) to improve final results (p. 8).        │
│  6. Noted limitations include the O(n^2 · d) time and memory cost of full self-attention for very long          │
│  sequences and the paper suggests restricted or local attention as future work (p. 6–7).                        │
│  7. The paper does not analyze domain-specific risks such as clinical mistranslation, privacy, or regulatory    │
│  concerns for healthcare use; domain adaptation and specialised evaluation would be required for sensitive      │
│  applications (p. 6–8).                                                                                         │
│  8. The authors recommend practical training and inference techniques (checkpoint averaging, regularization,    │
│  tuned beam/length-penalty) and indicate that applying